# T15 — DDL Import: SQL to Synthetic Data

Import SQL DDL (CREATE TABLE statements) and generate synthetic data from any existing database design.

**What you'll learn:**
- Parse SQL DDL into a Spindle schema
- Understand type-to-strategy mapping and column name heuristics
- Generate data from the inferred schema
- Verify FK integrity across generated tables

## 1. Define SQL DDL

In [ ]:
ddl = """
CREATE TABLE customer (
    customer_id INT IDENTITY(1,1) NOT NULL,
    first_name NVARCHAR(50) NOT NULL,
    last_name NVARCHAR(50) NOT NULL,
    email NVARCHAR(100),
    loyalty_tier VARCHAR(20),
    created_at DATETIME2,
    CONSTRAINT PK_customer PRIMARY KEY (customer_id)
);

CREATE TABLE product (
    product_id INT IDENTITY(1,1) NOT NULL,
    product_name NVARCHAR(100) NOT NULL,
    category VARCHAR(50),
    unit_price DECIMAL(10,2),
    is_active BIT DEFAULT 1,
    CONSTRAINT PK_product PRIMARY KEY (product_id)
);

CREATE TABLE [order] (
    order_id INT IDENTITY(1,1) NOT NULL,
    customer_id INT NOT NULL,
    order_date DATE NOT NULL,
    total DECIMAL(10,2),
    status VARCHAR(20),
    CONSTRAINT PK_order PRIMARY KEY (order_id),
    CONSTRAINT FK_order_customer FOREIGN KEY (customer_id)
        REFERENCES customer(customer_id)
);

CREATE TABLE order_line (
    line_id INT IDENTITY(1,1) NOT NULL,
    order_id INT NOT NULL,
    product_id INT NOT NULL,
    quantity INT,
    line_total DECIMAL(10,2),
    CONSTRAINT PK_order_line PRIMARY KEY (line_id),
    CONSTRAINT FK_line_order FOREIGN KEY (order_id) REFERENCES [order](order_id),
    CONSTRAINT FK_line_product FOREIGN KEY (product_id) REFERENCES product(product_id)
);
"""
print(f"DDL defines 4 tables with explicit FK constraints")

## 2. Parse DDL into Spindle Schema

In [ ]:
from sqllocks_spindle.schema.ddl_parser import DdlParser

parser = DdlParser()
schema = parser.parse_string(ddl)

print(f"Tables: {list(schema.tables.keys())}")
print(f"Relationships: {len(schema.relationships)}")
for r in schema.relationships:
    print(f"  {r.child}.{r.child_columns} -> {r.parent}.{r.parent_columns}")

## 3. Inspect Inferred Strategies

In [ ]:
for table_name, table_def in schema.tables.items():
    print(f"\n=== {table_name} ===")
    for col_name, col_def in table_def.columns.items():
        strategy = col_def.generator.get('strategy', 'unknown')
        print(f"  {col_name:20s} -> {strategy}")

## 4. Generate Synthetic Data

In [ ]:
from sqllocks_spindle import Spindle

result = Spindle().generate(schema=schema, scale="small", seed=42)
print(result.summary())

## 5. Verify FK Integrity

In [ ]:
errors = result.verify_integrity()
if errors:
    for e in errors:
        print(f"ERROR: {e}")
else:
    print("All FK relationships are valid!")

# Inspect generated data
print(f"\nCustomers: {len(result['customer'])} rows")
result['customer'].head()

## 6. CLI Equivalent

```bash
# Save DDL to file, then:
spindle from-ddl my_tables.sql --output my_schema.spindle.json
spindle generate --schema my_schema.spindle.json --scale small --output ./data/
```